In [1]:
import pandas as pd
import numpy as np
import re

# Vietnamese character set for Regex

In [2]:
viet_chars = (
    r'àáâãèéêìíòóôõùúýăđơư'
    r'ạảấầẩẫậắằẳẵặẹẻẽếềểễệ'
    r'ỉịọỏốồổỗộớờởỡợụủứừửữự'
    r'ỳỵỷỹÀÁÂÃÈÉÊÌÍÒÓÔÕÙÚÝĂĐƠƯ'
)

def remove_vietnamese(text):
    if not isinstance(text, str) or pd.isna(text):
        return text
    text = re.sub(rf'\s*\([^)]*[{viet_chars}][^)]*\)', '', text)   # remove (Vietnamese)
    text = re.sub(rf'\s*/[^/]*[{viet_chars}].*', '', text)          # remove / Vietnamese…
    text = re.sub(rf'\s+[{viet_chars}a-zA-Z]*[{viet_chars}][\w\s]*$', '', text)
    return text.strip()


# STEP 1 – Normalise headers, keep only valid columns

In [3]:
def clean_headers_and_columns(df):
    df_temp = df.copy()
    df_temp.columns = [str(c).strip() for c in df_temp.columns]

    # Drop fully-unnamed / NaN columns
    valid_cols = [c for c in df_temp.columns
                  if not str(c).lower().startswith('unnamed') and str(c).lower() != 'nan']
    df_temp = df_temp[valid_cols]

    cols = list(df_temp.columns)
    if len(cols) >= 2:
        cols[0], cols[1] = 'original_name', 'unit'
        df_temp.columns = cols
    return df_temp

# STEP 2 – Drop garbage rows

In [4]:
def filter_garbage_rows(df):
    df_temp = df.copy()

    # Must have a non-empty, non-NaN original_name
    df_temp = df_temp.dropna(subset=['original_name'])
    df_temp = df_temp[df_temp['original_name'].astype(str).str.strip() != '']
    df_temp = df_temp[df_temp['original_name'].astype(str).str.strip().str.lower() != 'nan']

    # Drop rows whose first cell looks like a section header
    # (no unit value AND all monthly columns are NaN)
    month_cols = [c for c in df_temp.columns
                  if str(c).strip() in ['Jan','Feb','Mar','Apr','May','Jun',
                                        'Jul','Aug','Sep','Oct','Nov','Dec']]
    if month_cols:
        all_nan_months = df_temp[month_cols].isnull().all(axis=1)
        no_unit = df_temp['unit'].isna() | (df_temp['unit'].astype(str).str.strip() == '')
        df_temp = df_temp[~(all_nan_months & no_unit)]

    # Extra: reject rows that start with obvious Vietnamese-only garbage labels
    garbage_pattern = r'^(Ko|Không|Ghi|Lưu|Note|Notes)\b'
    df_temp = df_temp[~df_temp['original_name'].astype(str)
                        .str.strip().str.match(garbage_pattern, case=False)]

    return df_temp.reset_index(drop=True)

# STEP 3 – Melt wide → long (months only)

In [5]:
def melt_data(df, category_name):
    df_temp = df.copy()
    df_temp.columns = df_temp.columns.astype(str).str.strip()

    months = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
    valid_months = [m for m in months if m in df_temp.columns]

    if not valid_months:
        return pd.DataFrame()

    df_long = df_temp.melt(
        id_vars=['original_name', 'unit'],
        value_vars=valid_months,
        var_name='period',
        value_name='value'
    )
    df_long.insert(0, 'category', category_name)
    return df_long

# STEP 4 – Strip Vietnamese from names

In [6]:
def normalize_names(df_long):
    df_temp = df_long.copy()
    df_temp.insert(1, 'name', df_temp['original_name'].apply(remove_vietnamese))
    df_temp['name'] = df_temp['name'].str.strip()
    return df_temp

# STEP 5 – Coerce values, add Month + has_data, impute NaN with median

In [7]:
def process_numeric_data(df_long):
    df_temp = df_long.copy()
    df_temp['unit'] = df_temp['unit'].astype(str).str.strip()

    # Coerce to numeric
    df_temp['value'] = (df_temp['value']
                        .astype(str).str.strip()
                        .str.replace(',', '', regex=False)
                        .replace(['-', '', 'nan', 'NaN', 'None'], np.nan))
    df_temp['value'] = pd.to_numeric(df_temp['value'], errors='coerce')

    # has_data = 1 if original cell had a real number
    has_data_series = df_temp['value'].notna().astype(int)

    month_map = {
        'Jan':1,'Feb':2,'Mar':3,'Apr':4,'May':5,'Jun':6,
        'Jul':7,'Aug':8,'Sep':9,'Oct':10,'Nov':11,'Dec':12
    }
    month_series = df_temp['period'].map(month_map)

    # Impute NaN with per-product median; fall back to 0
    df_temp['value'] = (df_temp
                        .groupby('name')['value']
                        .transform(lambda x: x.fillna(x.median())))
    df_temp['value'] = df_temp['value'].fillna(0).round(6)

    # Insert helper columns
    period_idx = df_temp.columns.get_loc('period')
    df_temp.insert(period_idx + 1, 'Month', month_series)

    value_idx = df_temp.columns.get_loc('value')
    df_temp.insert(value_idx + 1, 'has_data', has_data_series)

    return df_temp

Master pipeline có thể comment từng df để verify dữ liệu được xử lý như thế nào

In [13]:
# Master pipeline
def clean_table(df_raw, category_name):
    df1 = clean_headers_and_columns(df_raw)
    df2 = filter_garbage_rows(df1)
    df3 = melt_data(df2, category_name)
    if df3.empty:
        return df3
    df4 = normalize_names(df3)
    df5 = process_numeric_data(df4)
    # return df1
    # return df2
    # return df3
    # return df4
    return df5

In [14]:
file_names = ['Water_Recycle', 'Water_Withdrawal']
cleaned_tables = []

for name in file_names:
    path = f'./{name}.csv'
    try:
        df_raw = pd.read_csv(path)
        print(f'\n▶  Processing: {name}  (raw shape: {df_raw.shape})')
        print('   Raw preview:')
        print(df_raw.head(4).to_string())

        df_clean = clean_table(df_raw, name)

        if not df_clean.empty:
            cleaned_tables.append(df_clean)
            print(f'   ✔  Clean shape: {df_clean.shape}')
            print(df_clean.to_string())
        else:
            print('   ✖  Skipped – no valid month columns found.')

    except FileNotFoundError:
        print(f'   ✖  File not found: {path}')


▶  Processing: Water_Recycle  (raw shape: (7, 19))
   Raw preview:
                     Water Recycle unit   Jan   Feb   Mar   Apr   May   Jun   Jul   Aug   Sep   Oct   Nov   Dec  YTD       Q1       Q2       Q3       Q4
0  Water Recycle/ Nước tái sử dụng   m3  3047  2139  5669  5231  4453  4245  6172  5372  4704  4552  5187  5392  NaN  10855.0  13929.0  16248.0  15131.0
1                              NaN  NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN  NaN      NaN      NaN      NaN      NaN
2                  Ko có nước ngầm  NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN  NaN      NaN      NaN      NaN      NaN
3                              NaN  NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN  NaN      NaN      NaN      NaN      NaN
   ✔  Clean shape: (12, 8)
         category           name                    original_name unit period  Month  value  has_data
0   Water_Recycle  Water Recycle  Wate

In [15]:
if cleaned_tables:
    final_df = pd.concat(cleaned_tables, ignore_index=True)
    final_cols = ['category','name','original_name','unit','period','Month','value','has_data']
    existing_cols = [c for c in final_cols if c in final_df.columns]
    final_df = final_df[existing_cols]

    out_path = 'Water_Master_Cleaned.csv'
    final_df.to_csv(out_path, index=False)
    print(f'\n✅  Master file saved → {out_path}  (total rows: {len(final_df)})')
    print(final_df.to_string())


✅  Master file saved → Water_Master_Cleaned.csv  (total rows: 144)
             category                               name                                 original_name    unit period  Month        value  has_data
0       Water_Recycle                      Water Recycle               Water Recycle/ Nước tái sử dụng      m3    Jan      1  3047.000000         1
1       Water_Recycle                      Water Recycle               Water Recycle/ Nước tái sử dụng      m3    Feb      2  2139.000000         1
2       Water_Recycle                      Water Recycle               Water Recycle/ Nước tái sử dụng      m3    Mar      3  5669.000000         1
3       Water_Recycle                      Water Recycle               Water Recycle/ Nước tái sử dụng      m3    Apr      4  5231.000000         1
4       Water_Recycle                      Water Recycle               Water Recycle/ Nước tái sử dụng      m3    May      5  4453.000000         1
5       Water_Recycle                      W